In [2]:
import polars as pl

# Указываем пути (проверь, чтобы названия файлов совпадали)
batch_file = "../data/raw/batch_1.parquet"

print("Строим граф вычислений...")

# 1. SCAN (не Read!) - мы просто "смотрим" на файл, не загружая его в RAM
lazy_batch = pl.scan_parquet(batch_file)

# 2. Возьмем любой event_id для примера (допустим, событие № 24)
TARGET_EVENT = 24

# 3. Строим запрос (Query)
# Мы говорим: отфильтруй по ID -> забери результат в память (collect)
single_event = (
    lazy_batch
    .filter(pl.col("event_id") == TARGET_EVENT)
    .collect()  # <--- Только здесь происходит реальное чтение с диска!
)

print(f"Поймали нейтрино №{TARGET_EVENT}!")
print(f"Количество вспышек света: {single_event.shape[0]}")
print(single_event.head())



Строим граф вычислений...
Поймали нейтрино №24!
Количество вспышек света: 61
shape: (5, 5)
┌───────────┬──────┬────────┬───────────┬──────────┐
│ sensor_id ┆ time ┆ charge ┆ auxiliary ┆ event_id │
│ ---       ┆ ---  ┆ ---    ┆ ---       ┆ ---      │
│ i16       ┆ i64  ┆ f64    ┆ bool      ┆ i64      │
╞═══════════╪══════╪════════╪═══════════╪══════════╡
│ 3918      ┆ 5928 ┆ 1.325  ┆ true      ┆ 24       │
│ 4157      ┆ 6115 ┆ 1.175  ┆ true      ┆ 24       │
│ 3520      ┆ 6492 ┆ 0.925  ┆ true      ┆ 24       │
│ 5041      ┆ 6665 ┆ 0.225  ┆ true      ┆ 24       │
│ 2948      ┆ 8054 ┆ 1.575  ┆ true      ┆ 24       │
└───────────┴──────┴────────┴───────────┴──────────┘


In [3]:
# Загружаем геометрию (она маленькая, можно читать сразу)
geometry = pl.read_csv("../data/raw/sensor_geometry.csv")

# Делаем LEFT JOIN (слияние двух таблиц по колонке sensor_id)
# Это как склеить два множества в математике
full_event = single_event.join(
    geometry, 
    on="sensor_id", 
    how="left"
)

# Оставляем только нужные колонки для графа
final_nodes = full_event.select([
    "x", "y", "z", "time", "charge"
])

print("Готовая матрица признаков для графовой сети (Node Features):")
print(final_nodes.head())

Готовая матрица признаков для графовой сети (Node Features):
shape: (5, 5)
┌─────────┬────────┬────────┬──────┬────────┐
│ x       ┆ y      ┆ z      ┆ time ┆ charge │
│ ---     ┆ ---    ┆ ---    ┆ ---  ┆ ---    │
│ f64     ┆ f64    ┆ f64    ┆ i64  ┆ f64    │
╞═════════╪════════╪════════╪══════╪════════╡
│ 303.41  ┆ 335.64 ┆ 206.58 ┆ 5928 ┆ 1.325  │
│ -145.45 ┆ 374.24 ┆ 212.73 ┆ 6115 ┆ 1.175  │
│ 505.27  ┆ 257.88 ┆ -174.6 ┆ 6492 ┆ 0.925  │
│ -9.68   ┆ -79.5  ┆ 181.0  ┆ 6665 ┆ 0.225  │
│ 576.37  ┆ 170.92 ┆ 357.88 ┆ 8054 ┆ 1.575  │
└─────────┴────────┴────────┴──────┴────────┘
